📦 Bloque 0 — Imports estándar

In [ ]:
import pandas as pd
import numpy as np
import sys
import importlib
import json
import os
from os import path
from pathlib import Path

📦 Bloque 1 – Importación de librerías

En este bloque se importan las librerías necesarias para:
- Manipular archivos Excel y datos con pandas y numpy.
- Conectarse a SQL Server mediante SQLAlchemy.
- Trabajar con fechas, rutas de archivos y utilidades (copias temporales, esperas, regex, acentos).
    
Estas herramientas permiten la automatización completa del proceso de lectura, limpieza y carga de datos en SQL Server.

In [ ]:
# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:", CONFIG_DIR, "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"

try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)

    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))

except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"sys.path{sys.path[0] if sys.path else '<vacío>'}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "config_volvek_flujo.json")

try:
    with open(config_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()) if isinstance(data, dict) else type(data))

except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")

🔌 Bloque 2 – Conexión a SQL Server

Configura el engine de SQLAlchemy: 
- Parámetros: server, database, schema, table.
- Usa ODBC Driver 17 con Trusted Connection.
- Activa fast_executemany=True para cargas más rápidas.
    
Salida: objeto engine reutilizado en las consultas y en to_sql().

In [ ]:
# --- Parámetros desde config ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

# --- Construir engine + obtener connection_string pyodbc ---
driver = "ODBC Driver 17 for SQL Server"

# 1) ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# 2) SQLAlchemy engine → usando helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",
)

📁 Bloque 3 – Parámetros de entrada (carpeta y hoja)

Define:
- RUTA_ARCHIVOS: carpeta donde se buscará el Excel más reciente.
- EXTS = {".xlsx"}: extensiones válidas.
- Selección de hoja: intenta por nombre preferido y, si no existe, usa el primer índice.
    
Tip: si cambian los nombres de hojas en origen, ajusta PREFERRED_SHEETS.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "VOLVEK FLUJO").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx",)

PREFERRED_SHEETS = ["Base Flujo", "Base cesantia SURA Flujo", "Base"]
FALLBACK_SHEET_INDEX = 0

🔍 Bloque 4 — Detección automática de archivo y hoja + preview

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen   = info["archivo_origen"]
excel_path_used  = info["excel_path_used"]
_tmp_copy_path   = info["tmp_copy_path"]
target           = info["target_sheet"]
sheet_names      = info["sheet_names"]

# (Opcional) preview sin header
df_opexcel = fun.read_excel_safe_no_header(excel_path_used, target)
fun.pretty_table(df_opexcel, n=5)

📄 Archivo más reciente: Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx  |  2025-12-29 16:15:41
Hojas: ['Base Flujo']
✅ Hoja objetivo: Base Flujo


📊 Bloque 5 — Lectura + normalización de headers

In [ ]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
fun.pretty_table(df_raw, n=5)

Encabezados normalizados: ['noperacion', 'afinom', 'afirut', 'afirutdv', 'crecuotot', 'cresolmon', 'fecinicob', 'fectercob', 'crecuomon', 'prima_cliente', 'prima_exenta', 'comision_caja_25', 'comision_caja_variable_33', 'producto', 'folio_original', 'fecha_otorgamiento_folio_original']


,noperacion,afinom,afirut,afirutdv,crecuotot,cresolmon,fecinicob,fectercob,crecuomon,prima_cliente,prima_exenta,comision_caja_25,comision_caja_variable_33,producto,folio_original,fecha_otorgamiento_folio_original
0,032000036214,CATALINA CRISOSTOMO HENRIQUEZ,18857653,2,48,497370,20230703,20270731,19744,997,837.8151260504202,209.45378151260505,276.47899159663865,REPROGRAMACION,137000165328,20150331
1,130000147806,JUAN CUBILLOS GONZALEZ,13043029,5,44,2487195,20230224,20261031,65251,4983,4187.394957983193,1046.8487394957983,1381.8403361344538,REPROGRAMACION,001027635313,20150409
2,136000197543,DEIRA ESPINOSA VALENZUELA,14135760,3,80,4934040,20220328,20290131,97840,9886,8307.563025210084,2076.890756302521,2741.4957983193276,REPROGRAMACION,001027630836,20141219
3,122000757067,ANDREA PAZ CHEUQUEMAN CHEUQUEMAN,15583226,6,42,2085467,20250512,20281130,33358,4178,3510.924369747899,877.7310924369748,1158.6050420168067,REPROGRAMACION,122000733137,20141106
4,082001773079,JORGE TORO ISLA,9232042,1,55,1568556,20250703,20300228,39793,3143,2641.1764705882356,660.2941176470589,871.5882352941178,REPROGRAMACION,177000040127,20141211


🧩 Bloque 6 — Mapeo de columnas (Origen → Destino)

In [ ]:
# Origen → Destino
foliocredito        = fun.pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio")
NombreAfiliado      = fun.pick(df_raw, "afinom", "nombre_afiliado", "nombre")
rutafiliado         = fun.pick(df_raw, "afirut", "rut_afiliado", "rut")
dvafiliado          = fun.pick(df_raw, "afirutdv", "dv_afiliado", "dv")
Plazo               = fun.pick(df_raw, "crecuotot", "plazo")
MontoBruto          = fun.pick(df_raw, "cresolmon", "monto_bruto", "monto")
fechaPrimerVto      = fun.pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob")
FechaUltimoVto      = fun.pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob")
ValorCuota          = fun.pick(df_raw, "crecuomon", "valor_cuota")
PrimaBruta_src      = fun.pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_")
PrimaNeta_src       = fun.pick(df_raw, "prima_exenta", "prima_neta")
Com25_src           = fun.pick(df_raw, "comision_caja_25", "comision_caja_25_", "comision_25")
Com26_src           = fun.pick(df_raw, "comision_caja_variable_33", "comision_variable_33", "comision_33")
Producto            = fun.pick(df_raw, "producto")
FolioOrigen         = fun.pick(df_raw, "folio_original", "folio_origen")
FechaOrigen         = fun.pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen")

df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "NombreAfiliado": NombreAfiliado,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "PrimaBruta": PrimaBruta_src,
    "PrimaNeta": PrimaNeta_src,
    "Comision25": Com25_src,
    "ComisionVariable": Com26_src,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "Nombre_de_archivo": archivo_origen.name,
})

fun.pretty_table(df_can, n=5)

,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
0,032000036214,CATALINA CRISOSTOMO HENRIQUEZ,18857653,2,48,497370,20230703,20270731,19744,997,837.8151260504202,209.45378151260505,276.47899159663865,REPROGRAMACION,137000165328,20150331,Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx
1,130000147806,JUAN CUBILLOS GONZALEZ,13043029,5,44,2487195,20230224,20261031,65251,4983,4187.394957983193,1046.8487394957983,1381.8403361344538,REPROGRAMACION,001027635313,20150409,Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx
2,136000197543,DEIRA ESPINOSA VALENZUELA,14135760,3,80,4934040,20220328,20290131,97840,9886,8307.563025210084,2076.890756302521,2741.4957983193276,REPROGRAMACION,001027630836,20141219,Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx
3,122000757067,ANDREA PAZ CHEUQUEMAN CHEUQUEMAN,15583226,6,42,2085467,20250512,20281130,33358,4178,3510.924369747899,877.7310924369748,1158.6050420168067,REPROGRAMACION,122000733137,20141106,Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx
4,082001773079,JORGE TORO ISLA,9232042,1,55,1568556,20250703,20300228,39793,3143,2641.1764705882356,660.2941176470589,871.5882352941178,REPROGRAMACION,177000040127,20141211,Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx


🗂️ Bloque 7 — Canonicalización nombre de archivo

In [ ]:
nombre_canonico = fun.canonicalizar_nombre_archivo(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["Nombre_de_archivo"] = nombre_canonico

Nombre original : Base cesantia SURA Flujo NOVIEMBRE 2025.xlsx
Nombre canónico : Base cesantia SURA Flujo 202511.xlsx


🧹 Bloque 8 — Eliminación de filas finales nulas (trailing totals)

In [ ]:
df_can = fun.drop_trailing_mostly_null(
    df_can,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("PrimaBruta", "PrimaNeta", "Comision25", "ComisionVariable"),
    null_ratio_threshold=0.80,
    verbose=True,
)

🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…

🧾 Fila eliminada:


,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
56,,,56,,,,,,83910.625,430463,361733.61344537814,90433.40336134454,119372.0924369748,,,,Base cesantia SURA Flujo 202511.xlsx


⚙️ Bloque 9 — Casting de tipos + preparación df_sql

In [ ]:
# --- Casting numérico ---
fun.cast_numeric_columns(
    df_can,
    bigint_cols=["foliocredito", "FolioOrigen"],
    int_cols=["rutafiliado", "Plazo", "fechaPrimerVto", "FechaUltimoVto", "FechaOrigen"],
    float_cols=["MontoBruto", "ValorCuota", "PrimaBruta", "PrimaNeta", "Comision25", "ComisionVariable"],
)

# --- DV: char(1) mayúscula ---
fun.normalize_dv_column(df_can, "dvafiliado")

# --- Strings: trim y largo máximo ---
fun.trim_string_columns(df_can, {
    "NombreAfiliado": 510,
    "Producto": 510,
    "Nombre_de_archivo": 50,
})

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

# --- Nulos en columnas críticas ---
fun.report_nulls(df_can, ["foliocredito", "rutafiliado", "dvafiliado", "FolioOrigen", "Nombre_de_archivo"])

# --- Construir df_sql con orden de columnas para SQL ---
cols_sql = [
    "foliocredito", "NombreAfiliado", "rutafiliado", "dvafiliado", "Plazo", "MontoBruto",
    "fechaPrimerVto", "FechaUltimoVto", "ValorCuota", "PrimaBruta", "PrimaNeta",
    "Comision25", "ComisionVariable", "Producto", "FolioOrigen", "FechaOrigen", "Nombre_de_archivo",
]

df_sql = fun.build_sql_frame(df_can, cols_sql)

print("\n📊 df_sql listo con columnas:", list(df_sql.columns))
fun.pretty_table(df_sql, n=5)

✅ dtypes DESPUÉS:
 foliocredito                  Int64
NombreAfiliado       string[python]
rutafiliado                   Int64
dvafiliado                   object
Plazo                         Int64
MontoBruto                  float64
fechaPrimerVto                Int64
FechaUltimoVto                Int64
ValorCuota                  float64
PrimaBruta                  float64
PrimaNeta                   float64
Comision25                  float64
ComisionVariable            float64
Producto             string[python]
FolioOrigen                   Int64
FechaOrigen                   Int64
Nombre_de_archivo    string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafiliado: 0 nulos
 - dvafiliado: 0 nulos
 - FolioOrigen: 0 nulos
 - Nombre_de_archivo: 0 nulos

📦 df_sql listo con columnas: ['foliocredito', 'NombreAfiliado', 'rutafiliado', 'dvafiliado', 'Plazo', 'MontoBruto', 'fechaPrimerVto', 'FechaUltimoVto', 'ValorCuota', 'PrimaBruta', 'PrimaNeta', 'Com

🧾 Bloque 10 — Extracción de valores de partición

In [ ]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚿 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - Base cesantia SURA Flujo 202511.xlsx: 56 filas


🚀 Bloque 11 — Carga a SQL Server (replace_partition / append)

In [ ]:
resumen = []

# Nombre lógico de tabla desde config
table_name = data["tablas_remotas"]["volvek_acumulado_flujo"]

for nombre_archivo in vals:

    # 1) Subconjunto del DataFrame
    df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

    if df_sub.empty:
        print(f"⚠️ No se encontraron filas para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
        continue

    # 2) Conteo previo en destino (COUNT *)
    sql_count = f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table_name}
        WHERE Nombre_de_archivo = '{nombre_archivo}'
    """

    df_cnt = fun.query_to_df(
        sql=sql_count,
        connection_string=connection_string,
        engine="pandas",
        return_iter=False,
    )

    existing_count = int(df_cnt.iloc[0, 0]) if not df_cnt.empty else 0

    # 3) Decisión: reemplazo vs inserción nueva
    if existing_count > 0:
        print(
            f"♻️ Se encontró data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}' ({existing_count} filas en {table_name})."
        )
        print("🧹 Eliminando filas anteriores para reemplazarlas...")

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="replace_partition",
            partition_column="Nombre_de_archivo",
            partition_values=[nombre_archivo],
            engine="pandas",
        )

        accion = "reemplazo"
        filas_borradas = summary["rows_deleted"]

        print(
            f"✅ Filas eliminadas en destino para "
            f"'{nombre_archivo}': {filas_borradas}"
        )

    else:
        print(
            f"🆕 No hay data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}'. Se insertará como archivo nuevo."
        )

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="append",
            engine="pandas",
        )

        accion = "inserción nueva"

    # 4) Log y resumen
    filas_sub = len(df_sub)
    print(
        f"📥 Insertadas {filas_sub} filas para "
        f"Nombre_de_archivo = '{nombre_archivo}'."
    )

    resumen.append(
        (nombre_archivo, filas_sub, existing_count, accion)
    )

# 5) Resumen final
print("\n📈 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(
            f"   - {nombre_archivo}: {filas_sub} filas cargadas "
            f"(reemplazando {existing_count} previas)."
        )
    else:
        print(
            f"   - {nombre_archivo}: {filas_sub} filas cargadas "
            f"(archivo nuevo)."
        )

🆕 No hay data previa para Nombre_de_archivo = 'Base cesantia SURA Flujo 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 56 filas para Nombre_de_archivo = 'Base cesantia SURA Flujo 202511.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - Base cesantia SURA Flujo 202511.xlsx: 56 filas cargadas (archivo nuevo).


🗑️ Bloque 12 — Eliminación de archivo origen

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK FLUJO\Base cesantia SURA Flujo OCTUBRE 2025.xlsx


# SQL

### Bloque 13 — Query: archivos cargados

In [ ]:
query_1 = f"""
-- [VOLVEK_FLUJO] Archivos cargados (desc)
SELECT DISTINCT Nombre_de_archivo
FROM {data["tablas_remotas"]["volvek_acumulado_flujo"]}
ORDER BY Nombre_de_archivo DESC;
"""

df_q1 = fun.query_to_df(
    sql=query_1,
    connection_string=connection_string,
    engine="pandas",
)

fun.pretty_table(df_q1)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

### Bloque 14 — Query: DROP tabla final

In [ ]:
query_2 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["flujo_final_acumulado"]};
"""

fun.exec_sql(query_2, connection_string)

### Bloque 15 — Query: SELECT INTO tabla final

In [ ]:
query_3 = f"""
SELECT *,
       4659577 AS POLIZA,
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       4277 AS PLAN_TECNICO,
       4 AS PLAZO_CUOTAS,
       'Credito Consumo' AS Negocio
INTO {data["tablas_remotas"]["flujo_final_acumulado"]}
FROM {data["tablas_remotas"]["volvek_acumulado_flujo"]};
"""

fun.exec_sql(query_3, connection_string)